# 🔍 Evaluator-Optimizer: Generate, Critique, Refine

The **evaluator-optimizer** workflow generates a candidate, grades it against
explicit criteria, and feeds the critique back into a regeneration — looping
until the grade passes.

This notebook rebuilds the joke generator from
`05_AI_Agent_Fundamentals/4. Workflow_Pattern/5. Evaluator_Optimizer/` — same
evaluation schema, same criteria, same topic — twice: once as a plain LCEL loop,
and once with LangChain 1.x agent middleware, which is the most natural home this
pattern has in the 1.x API.

```
topic ──▶ generate ──▶ evaluate ──┬── approved ──────────▶ final
             ▲                    │
             └── feedback ────────┘  (bounded, or it never ends)
```

## Learning Objectives
In this notebook, you will learn:
1. **The cycle problem** - why a pure LCEL chain cannot loop back on itself
2. **Structured grading** - constrain the evaluator to a verdict plus actionable feedback
3. **The LCEL loop** - drive the cycle from Python and keep the chains declarative
4. **Middleware loops** - use `after_model` with `jump_to="model"` for a native 1.x cycle
5. **Bounding the loop** - why `ModelCallLimitMiddleware` is not optional

## Prerequisites
- API credentials in a `.env` file at the repo root, for whichever provider the `helpers` factory selects on your platform
- `langchain >= 1.0`, `langchain-core >= 1.0`, `pydantic >= 2`
- Completion of `6.4_Orchestrator_Worker.ipynb`

---

## 🔧 1. Setting Up the Environment

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API Keys & Import Dependencies
# ============================================================================
# We use python-dotenv to securely load API keys from a .env file
# This is a best practice - never hardcode API keys in your notebooks!
# ============================================================================

import warnings

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, after_model
from langchain.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing_extensions import Literal

warnings.filterwarnings("ignore")

# Load environment variables from .env file
load_dotenv()

# temperature=0.7 — the generator needs room to vary between drafts, otherwise
# every refinement pass returns the same joke and the loop cannot converge.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

print("✅ Environment variables loaded successfully!")
print(f"🤖 LLM initialized: {llm.model_name}")

# ----------------------------------------------------------------------------
# PREVIOUS SETUP (kept for reference): platform-aware `helpers` factory
# ----------------------------------------------------------------------------
# import os
# import platform
# import sys
#
# from helpers.utils import get_databricks_llm, get_groq_llm, get_openai_llm
#
# print(f"📍 Running on: {platform.system()}")
#
# if sys.platform == "win32":
#     llm = get_groq_llm(temperature=0.7)                                    # Windows
# elif sys.platform == "darwin":
#     llm = get_databricks_llm("databricks-gemini-2-5-pro", temperature=0.7) # macOS
# else:
#     llm = get_groq_llm(temperature=0.7)                                    # Linux

---

## 🔁 2. The Cycle Problem

Every pattern so far mapped onto an LCEL operator because every pattern was a
**directed acyclic** flow. This one is not.

`a | b | a` does not loop. It builds a three-step chain that runs `a` twice. LCEL
has no operator that says "go back and try again, conditionally, until a
predicate passes". The `.with_retry()` trick from notebook 6.1 came close, but it
retries with **no memory of why it failed** — and this pattern's whole point is
feeding the critique forward.

So there are two honest options, and we build both:

| Approach | The loop lives in | Use when |
|---|---|---|
| LCEL plus a Python `while` | your driver function | the loop is orchestration you own |
| `create_agent` plus `after_model` | the agent graph, via `jump_to` | you want a single invocable object |

---

## 📋 3. The Evaluation Schema

Identical to the LangGraph version. The verdict is a `Literal` so it can be
tested with `==`, and the feedback is a separate field so it can be interpolated
into the next generation prompt.

Two fields, not one: a grade you can branch on, and a critique the generator can
act on. A single free-text field would give you neither reliably.

In [ ]:
# ============================================================================
# EVALUATION SCHEMA: A testable verdict plus actionable feedback
# ============================================================================


class ContentEvaluation(BaseModel):
    """Structured output schema for content evaluation."""

    quality_grade: Literal["approved", "needs_improvement"] = Field(
        description="Determine if the content meets quality standards or needs improvement"
    )
    improvement_feedback: str = Field(
        description="If content needs improvement, provide specific actionable feedback"
    )


evaluator = ChatPromptTemplate.from_template(
    "Evaluate the following joke for quality and humor:\n\n"
    "Joke: {joke}\n\n"
    "Consider these criteria:\n"
    "1. Is it genuinely funny or clever?\n"
    "2. Is it appropriate and inoffensive?\n"
    "3. Does it make logical sense?\n"
    "4. Is it well-structured with a clear setup and punchline?\n\n"
    "If it doesn't meet these standards, provide specific feedback for improvement."
) | llm.with_structured_output(ContentEvaluation)

print("✅ Evaluator ready")

---

## ✍️ 4. The Generator

One chain, two prompts. The first pass has no feedback to work with; every later
pass does. Keeping this as a single chain with an optional `{feedback}` slot is
simpler than maintaining two near-identical chains.

In [ ]:
# ============================================================================
# GENERATOR CHAIN: Writes a joke, optionally incorporating feedback
# ============================================================================
generator = ChatPromptTemplate.from_template(
    "Create a well-crafted, humorous joke about {topic}.\n\n"
    "{feedback_block}"
) | llm | StrOutputParser()


def feedback_block(feedback: str | None) -> str:
    """Render the feedback section of the prompt, or nothing on the first pass."""
    if not feedback:
        return ""
    return (
        f"Incorporate this feedback from a reviewer:\n{feedback}\n\n"
        "Make sure to address the specific points mentioned."
    )


print("✅ Generator ready")

---

## 🐍 5. Approach A: The LCEL Loop

The chains stay declarative; a small driver function owns the cycle. This is the
most direct translation of the LangGraph graph, and it makes the loop bound
impossible to overlook.

> **Note**: the original LangGraph notebook has **no iteration cap** — a
> stubborn evaluator loops forever. `MAX_ITERATIONS` here fixes that.

In [ ]:
# ============================================================================
# LCEL LOOP: Generate and evaluate until approved, or the cap is reached
# ============================================================================
MAX_ITERATIONS = 5


def run_evaluator_optimizer(topic: str) -> dict:
    """Generate, critique and refine until the evaluator approves."""
    print(f"Starting evaluator-optimizer workflow for topic: {topic}")
    feedback = None

    for iteration in range(1, MAX_ITERATIONS + 1):
        joke = generator.invoke({"topic": topic, "feedback_block": feedback_block(feedback)})
        verdict = evaluator.invoke({"joke": joke})

        print(f"\n🔄 Iteration {iteration}: {verdict.quality_grade}")
        print(f"   Draft: {joke[:100]}...")

        if verdict.quality_grade == "approved":
            print(f"✅ Approved after {iteration} iteration(s)")
            return {
                "content_topic": topic,
                "generated_content": joke,
                "quality_assessment": verdict.quality_grade,
                "iterations": iteration,
            }

        print(f"   Feedback: {verdict.improvement_feedback[:120]}...")
        feedback = verdict.improvement_feedback

    print(f"⚠️  Stopped at the {MAX_ITERATIONS}-iteration cap without approval")
    return {
        "content_topic": topic,
        "generated_content": joke,
        "quality_assessment": verdict.quality_grade,
        "iterations": MAX_ITERATIONS,
    }

In [ ]:
# ============================================================================
# TEST A: Run the LCEL loop on the same topic as the LangGraph notebook
# ============================================================================
result = run_evaluator_optimizer("Agentic AI systems")

print("\n" + "=" * 50)
print("EVALUATOR-OPTIMIZER WORKFLOW RESULTS")
print("=" * 50)
print(f"Topic: {result['content_topic']}")
print(f"Iterations: {result['iterations']}")
print(f"Quality Status: {result['quality_assessment']}")
print(f"\nFinal Content:\n{result['generated_content']}")

---

## 🧩 6. Approach B: The Middleware Loop

LangChain 1.x can express the cycle **inside** a single agent. The `after_model`
hook runs once the model has answered, and returning `jump_to="model"` sends
control back for another pass. That is a real conditional edge in the agent
graph, not a Python loop around it.

Three details make it work:

- `can_jump_to=["model"]` must be declared on the decorator. It is what creates
  the conditional edge; without it the jump is ignored.
- Appending a `HumanMessage` with the critique is how the feedback reaches the
  next generation. The message history **is** the state.
- Returning `None` means "no jump" — the agent falls through and finishes.

In [ ]:
# ============================================================================
# MIDDLEWARE: Evaluate after each model call, loop back when not approved
# ============================================================================
review_log: list[str] = []


@after_model(can_jump_to=["model"])
def evaluate_and_refine(state, runtime):
    """Grade the latest draft; jump back to the model when it needs work."""
    last = state["messages"][-1]

    # Only grade a finished assistant answer, never a tool-calling turn.
    if not isinstance(last, AIMessage) or getattr(last, "tool_calls", None):
        return None

    draft = last.text if hasattr(last, "text") else str(last.content)
    verdict = evaluator.invoke({"joke": draft})
    review_log.append(f"{verdict.quality_grade}: {draft[:60]}...")

    if verdict.quality_grade == "approved":
        print(f"✅ Approved: {draft[:80]}...")
        return None  # no jump — the agent ends here

    print(f"🔄 Needs improvement: {verdict.improvement_feedback[:100]}...")
    return {
        "messages": [
            HumanMessage(
                content=(
                    f"A reviewer graded that joke as needing improvement.\n"
                    f"Feedback: {verdict.improvement_feedback}\n\n"
                    f"Rewrite the joke addressing this feedback."
                )
            )
        ],
        "jump_to": "model",
    }


print("✅ Middleware defined")

The call-limit middleware is the bound. Without it a never-satisfied evaluator
loops until you kill the kernel. `exit_behavior="end"` stops cleanly and returns
the last draft rather than raising.

In [ ]:
# ============================================================================
# AGENT: Generator plus evaluator loop plus a hard iteration cap
# ============================================================================
joke_agent = create_agent(
    model=llm,
    tools=[],  # no tools — the loop is the whole mechanism
    system_prompt="You are a comedy writer. Write one well-crafted, clean joke on the given topic.",
    middleware=[
        evaluate_and_refine,
        ModelCallLimitMiddleware(run_limit=MAX_ITERATIONS, exit_behavior="end"),
    ],
)

print("✅ Agent assembled")

In [ ]:
# ============================================================================
# TEST B: Same topic, same criteria, run through the agent
# ============================================================================
review_log.clear()

response = joke_agent.invoke(
    {"messages": [HumanMessage(content="Write a joke about Agentic AI systems")]}
)

print("\n" + "=" * 50)
print("MIDDLEWARE LOOP RESULTS")
print("=" * 50)
print(f"Model calls: {len(review_log)}")
for entry in review_log:
    print(f"  - {entry}")
print(f"\nFinal Content:\n{response['messages'][-1].text}")

---

## ⚖️ 7. Choosing Between the Two

Both loops are bounded, both feed critique forward, and both produce the same
result. They differ in what you get around the loop.

| | LCEL loop | Middleware loop |
|---|---|---|
| The loop is | a Python `for` you can read at a glance | a conditional edge inside the agent |
| Bounded by | your `range(MAX_ITERATIONS)` | `ModelCallLimitMiddleware` |
| Result is | a plain dict you shaped | an agent state with full message history |
| Tracing | two separate chain runs per pass | one agent run, all passes nested |
| Adding tools | you would rebuild the loop by hand | already supported — the agent has a tool node |
| Streaming, checkpointing, interrupts | none | inherited from the agent runtime |

Start with the LCEL loop when the generator is a single call and the loop is
orchestration you own. Move to the middleware loop when the generator needs
tools, or when you want one invocable object with the agent runtime's streaming
and persistence.

---

## 📝 Summary

We rebuilt the LangGraph evaluator-optimizer twice, and this is the one pattern
where LCEL alone is not enough.

### 1. The translation table

| LangGraph | LangChain 1.x |
|---|---|
| `add_conditional_edges` back to `generate_content` | a Python loop, **or** `after_model` returning `jump_to="model"` |
| `WorkflowState` carrying feedback between passes | a local variable, **or** a `HumanMessage` appended to history |
| `with_structured_output(ContentEvaluation)` | identical — unchanged in 1.x |
| no iteration cap (a bug in the original) | `MAX_ITERATIONS`, **or** `ModelCallLimitMiddleware(run_limit=N)` |

### 2. What to remember
- **LCEL is acyclic.** `a | b | a` runs `a` twice, it does not loop. Cycles come
  from a Python loop, from agent middleware, or from LangGraph.
- **`.with_retry()` is not this pattern.** It retries blind. Evaluator-optimizer
  exists to feed the critique forward.
- **Always bound the loop.** A structured verdict does not guarantee the
  evaluator will ever say "approved".
- **`can_jump_to` is mandatory** on the decorator. It is what builds the
  conditional edge.

### 3. The five patterns, complete

| Pattern | LangChain 1.x primitive | Full coverage |
|---|---|---|
| Prompt chaining | `\|` plus `.with_retry()` plus `.with_fallbacks()` | yes |
| Routing | `RunnableBranch` | yes |
| Parallelization | `RunnableParallel` | yes |
| Orchestrator-worker | `.map()` | fan-out yes; shared state and per-worker durability need `Send` |
| Evaluator-optimizer | Python loop, or `after_model` plus `jump_to` | yes |

### Next Steps
- Read `README.md` in this folder for the full LangGraph-to-LangChain mapping
- Compare against the LangGraph originals in
  `05_AI_Agent_Fundamentals/4. Workflow_Pattern/`
- For cycles that need checkpointing, interrupts or human-in-the-loop, continue to
  `03_LangGraph_Fundamentals/`